
# MTA Daily Ridership — Week 3 Time Series Forecasting

## 1. Project Overview

This notebook builds a reproducible daily forecasting pipeline for the seven MTA transportation systems using the validated Week 1 cleaned dataset. Each system is forecast independently because units differ: estimated riders, scheduled trips, and vehicles are never combined.

Objectives:
- Forecast daily activity 30 days beyond 2024-10-31, with a 7-day horizon evaluated as well.
- Establish baselines (naive, weekly seasonal naive, moving average, drift) before advanced models.
- Backtest with chronological rolling origins; never shuffle or randomly split time series.
- Compare models with MAE, RMSE, WAPE, sMAPE and MASE, using MASE as the primary selection metric.
- Quantify forecast uncertainty with empirical residual intervals and check their coverage.
- Identify which systems have predictable daily patterns and which remain difficult to forecast.

Recovery percentage is deliberately **not** a forecasting target. It is a comparison metric affected by the February 2023 methodology change and by the 2023-11-10 one-day anomaly identified in Week 2.


In [1]:

import json
import os
import sys
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import acf, pacf


def locate_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "WEEK THREE" / "python").is_dir() and (
            candidate / "WEEK TWO" / "python" / "analysis_utils.py"
        ).is_file():
            return candidate
    raise FileNotFoundError("Run from inside the repository or set MTA_PROJECT_ROOT.")


_env_root = os.environ.get("MTA_PROJECT_ROOT")
PROJECT_ROOT = locate_root(Path(_env_root) if _env_root else Path.cwd()).resolve()
sys.path.insert(0, str(PROJECT_ROOT / "WEEK THREE" / "python"))

import forecasting_utils as fu

fu.w2.configure_plotting()
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 190)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

print(f"Project root: {PROJECT_ROOT.name}")
print(f"pandas {pd.__version__} | numpy {np.__version__}")
import sklearn, statsmodels
print(f"scikit-learn {sklearn.__version__} | statsmodels {statsmodels.__version__}")
print(f"Horizons: {fu.HORIZONS} | final test days: {fu.TEST_SIZE} | seed: {fu.SEED}")


Project root: MTA Project
pandas 2.3.3 | numpy 2.3.1
scikit-learn 1.9.1 | statsmodels 0.15.0
Horizons: (7, 30) | final test days: 60 | seed: 42



## 2. Data Loading and Validation

The loader reads the Week 1 cleaned CSV through Week 2's validated utility. Each daily series must have a complete, unique, gap-free DatetimeIndex, no missing values, and no negative activity. Genuine zero activity — notably the Staten Island Railway weekend zeros — is retained.


In [2]:

series = fu.load_daily_series()
validation = pd.DataFrame([fu.validate_series(s, name) for name, s in series.items()])
display(validation.to_string(index=False))
assert len(series) == 7
assert all(row["gap_days"] == 0 for _, row in validation.iterrows())
print("PASS: seven complete daily series; zeros retained, no missing or negative values.")


'            transport  rows   date_min   date_max  calendar_days  gap_days  duplicate_dates  missing_values  negative_values  zero_values\n              Subways  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\n                Buses  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\n                 LIRR  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\n          Metro-North  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\n        Access-A-Ride  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\n  Bridges and Tunnels  1706 2020-03-01 2024-10-31           1706         0                0               0                0            0\nStaten Island Railway  170

PASS: seven complete daily series; zeros retained, no missing or negative values.



## 3. Time Series Preparation

The final 60 days are held out as an untouched test set. All model selection uses rolling-origin backtests strictly before that window. Origins are identical across models and systems for fair comparison.


In [3]:

reference = series["Subways"]
split_index = len(reference) - fu.TEST_SIZE
split_date = reference.index[split_index]
print(f"Full history: {reference.index[0].date()} to {reference.index[-1].date()}")
print(f"Development ends: {reference.index[split_index - 1].date()}")
print(f"Final test starts: {split_date.date()}")

for horizon in fu.HORIZONS:
    origins = fu.make_origins(reference, horizon)
    print(f"h={horizon}: {len(origins)} origins, {origins[0].date()} to {origins[-1].date()}, "
          f"test {reference.index[-horizon].date()} to {reference.index[-1].date()}")

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(reference.index, reference.values, lw=1, color="#4c78a8", label="Subways activity")
ax.axvline(reference.index[split_index - 1], color="black", ls="--", lw=1.2, label="Development / test boundary")
ax.set_title("Chronological split — final 60 days held out from model selection")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated riders (Subways only)")
ax.legend(loc="upper left")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
fig.tight_layout()
fu.VIZ_DIR.mkdir(parents=True, exist_ok=True)
fig.savefig(fu.VIZ_DIR / "00_split_boundary.png", bbox_inches="tight")
plt.close(fig)


Full history: 2020-03-01 to 2024-10-31
Development ends: 2024-09-01
Final test starts: 2024-09-02
h=7: 6 origins, 2024-07-21 to 2024-08-25, test 2024-10-25 to 2024-10-31
h=30: 4 origins, 2024-05-04 to 2024-08-02, test 2024-10-02 to 2024-10-31



## 4. Exploratory Time Series Analysis

Daily series are shown with 7-day and 30-day rolling means. Small multiples keep each system on its own unit scale. The COVID-era collapse and the later recovery are visible in every mode, but the shape differs sharply by system.


In [4]:

fig, axes = plt.subplots(4, 2, figsize=(16, 11), sharex=True)
for ax, name in zip(axes.ravel(), fu.SYSTEM_NAMES):
    s = series[name]
    ax.plot(s.index, s.values, lw=0.6, alpha=0.45, color="grey")
    ax.plot(s.index, s.rolling(7, min_periods=7).mean(), lw=1.2, color=fu.w2.SYSTEM_COLORS[name], label="7-day mean")
    ax.plot(s.index, s.rolling(30, min_periods=30).mean(), lw=1.4, ls="--", color="black", label="30-day mean")
    ax.set_title(f"{name} ({fu.SYSTEM_UNITS[name]})")
    ax.xaxis.set_major_locator(mdates.YearLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
axes[0, 0].legend(fontsize=8)
axes.ravel()[-1].axis("off")
fig.suptitle("Daily activity with 7-day and 30-day rolling means")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(fu.VIZ_DIR / "01_daily_series_rolling.png", bbox_inches="tight")
plt.close(fig)

diagnostics = pd.DataFrame([fu.series_diagnostics(s) for name, s in series.items()])
display(diagnostics.round(3).to_string(index=False))


'            transport  days           mean            std     cv          min            max  zero_days  snaive_scale  acf_lag_1  acf_lag_7  acf_lag_14  acf_lag_28  acf_lag_365     covid_mean    recent_mean\n              Subways  1706 2,509,054.9790 1,062,183.6810 0.4230 198,399.0000 5,498,809.0000          0  203,333.3110     0.8000     0.9300      0.8910      0.8600       0.5770 1,093,821.1740 3,232,766.8960\n                Buses  1706 1,006,868.0930   440,379.9200 0.4370   5,498.0000 2,244,515.0000          0   92,992.4370     0.7630     0.9130      0.8610      0.7990       0.2920   356,021.7720 1,102,412.7570\n                 LIRR  1706   135,960.1160    71,298.7790 0.5240   1,903.0000   321,569.0000          0   10,899.6450     0.7800     0.9520      0.9280      0.9040       0.6410    47,222.6630   201,237.3390\n          Metro-North  1706   114,888.2730    66,500.2080 0.5790   3,281.0000   249,585.0000          0    9,773.5490     0.8270     0.9570      0.9450      0.9160    


## 5. Seasonality and Trend Diagnostics

Weekly seasonality (lag 7) is expected to be strong for transit modes. ACF and PACF describe linear dependence; the August 2026 ADF-style stationarity question is addressed informally through rolling means because the series contain a structural COVID break. Diagnostics are used only to justify model families, not to delete observations.


In [5]:

# Weekly pattern, indexed to each system's own mean (unit-free)
weekly = pd.DataFrame({name: series[name].groupby(series[name].index.dayofweek).mean() for name in fu.SYSTEM_NAMES})
weekly_indexed = 100 * weekly / weekly.mean()
fig, ax = plt.subplots(figsize=(11, 5))
for name in fu.SYSTEM_NAMES:
    ax.plot(range(7), weekly_indexed[name], marker="o", lw=1.4, label=name, color=fu.w2.SYSTEM_COLORS[name])
ax.set_xticks(range(7), ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"])
ax.axhline(100, color="grey", ls=":")
ax.set_title("Weekly seasonality index (100 = system's own daily mean)")
ax.set_xlabel("Day of week")
ax.set_ylabel("Activity index")
ax.legend(ncol=4, fontsize=8)
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "02_weekly_seasonality.png", bbox_inches="tight")
plt.close(fig)

seasonality = fu.w2.seasonality_index(
    fu.w2.build_long(fu.w2.load_cleaned())
)
fig, ax = plt.subplots(figsize=(12, 6))
image = ax.imshow(seasonality.values, cmap="YlGnBu", aspect="auto")
ax.set_xticks(range(seasonality.shape[1]), seasonality.columns)
ax.set_yticks(range(seasonality.shape[0]), seasonality.index)
for r in range(seasonality.shape[0]):
    for c in range(seasonality.shape[1]):
        value = seasonality.iloc[r, c]
        ax.text(c, r, f"{value:.0f}", ha="center", va="center", fontsize=8)
fig.colorbar(image, ax=ax, label="Activity index (100 = that year's mean)")
ax.set_title("Month-of-year pattern (context for annual seasonality)")
ax.set_xlabel("Month")
ax.set_ylabel("System")
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "03_monthly_seasonality.png", bbox_inches="tight")
plt.close(fig)

lags = np.arange(1, 29)
fig, axes = plt.subplots(4, 2, figsize=(15, 11), sharex=True)
for ax, name in zip(axes.ravel(), fu.SYSTEM_NAMES):
    values = series[name].values
    acf_values = acf(values, nlags=28, fft=True)[1:]
    pacf_values = pacf(values, nlags=28, method="ols")[1:]
    ax.bar(lags - 0.2, acf_values, width=0.4, alpha=0.7, label="ACF", color="#4c78a8")
    ax.bar(lags + 0.2, pacf_values, width=0.4, alpha=0.7, label="PACF", color="#f58518")
    ax.axhline(0, color="black", lw=0.8)
    ax.set_title(name)
axes[0, 0].legend(fontsize=8)
axes.ravel()[-1].axis("off")
fig.suptitle("ACF and PACF by system (lags 1-28)")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(fu.VIZ_DIR / "04_acf_pacf.png", bbox_inches="tight")
plt.close(fig)



## 6. Baseline Models

Four transparent baselines are fitted for every system and horizon: last value (naive), value from the same weekday last week (seasonal naive), 7-day moving average, and drift extending the historical slope. These set the minimum bar; advanced models must beat the seasonal naive baseline on MASE to be useful.


In [6]:

example = series["Subways"]
train = example.iloc[:-30]
actual = example.iloc[-30:]
baseline_models = {k: v for k, v in fu.model_registry(include_advanced=False).items()}
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(actual.index, actual.values, color="black", lw=1.6, label="Actual")
for name, fn in baseline_models.items():
    prediction = fu.safe_forecast(fn(train, 30, actual.index))
    ax.plot(actual.index, prediction, lw=1.2, label=name)
ax.set_title("Baseline forecasts on the final 30 days — Subways")
ax.set_xlabel("Date")
ax.set_ylabel("Estimated riders")
ax.legend(ncol=3, fontsize=8)
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "05_baselines_example.png", bbox_inches="tight")
plt.close(fig)
print("Baselines:", ", ".join(baseline_models))


Baselines: Naive, SeasonalNaive, MovingAverage7, Drift



## 7. Advanced Forecasting Models

Three model families are evaluated with the same origins as the baselines:

- **ETS / Holt-Winters** with additive trend and weekly seasonality.
- **SARIMA** `(1,1,1) x (0,0,1,7)` without a constant.
- **HistGBM**, a scikit-learn gradient-boosting regressor with lag 1/7/14/28, rolling mean/std, and calendar features. Features are built from past values only; multi-step forecasts are recursive. Random seed is fixed.

Any model that fails to fit at an origin is recorded as such and excluded from that comparison. Forecasts are clipped at zero because activity cannot be negative.


In [7]:

advanced = {k: v for k, v in fu.model_registry().items() if k not in baseline_models}
rows = []
for name, fn in advanced.items():
    try:
        prediction = fu.safe_forecast(fn(train, 30, actual.index))
        metrics = fu.compute_metrics(actual.values, prediction, train.values)
        rows.append({"model": name, "status": "OK", **metrics})
    except Exception as exc:
        rows.append({"model": name, "status": f"FAILED: {type(exc).__name__}: {exc}"})
display(pd.DataFrame(rows).to_string(index=False))
print("Feature example (last training row):")
display(fu._feature_frame(train).tail(1))


'  model status          MAE         RMSE   WAPE   sMAPE   MASE\n    ETS     OK 178,675.5199 220,198.6407 0.0491  4.9499 0.8790\n SARIMA     OK 722,736.5582 788,706.5371 0.1985 21.0526 3.5554\nHistGBM     OK 245,085.3671 314,548.6263 0.0673  6.5454 1.2057'

Feature example (last training row):


,lag_1,lag_7,lag_14,lag_28,roll_mean_7,roll_mean_14,roll_mean_28,roll_std_7,dayofweek,dayofmonth,month,dayofyear,weekofyear,is_weekend
Date,,,,,,,,,,,,,,
2024-10-01,"3,849,138.0000","4,323,708.0000","4,189,311.0000","3,629,404.0000","3,609,606.0000","3,618,072.9286","3,557,212.8929","958,088.6629",1,1,10,275,40,0



## 8. Rolling-Origin Backtesting

Backtests use 6 weekly-spaced origins for the 7-day horizon and 4 monthly-spaced origins for the 30-day horizon, all inside the development window. Every model sees identical origins and actuals, so comparisons are fair. Failures are captured rather than silently dropped.


In [8]:

backtest_parts, residual_parts = [], []
for name, s in series.items():
    for horizon in fu.HORIZONS:
        summary, residuals = fu.backtest_series(s, horizon)
        backtest_parts.append(summary)
        residual_parts.append(residuals)
    print(f"backtested {name}")

backtest_df = pd.concat(backtest_parts, ignore_index=True)
residuals_df = pd.concat(residual_parts, ignore_index=True)
backtest_df.to_csv(fu.REPORT_DIR / "backtesting_results.csv", index=False)
print("rows:", len(backtest_df), "successful:", int((backtest_df["status"] == "OK").sum()))
display(backtest_df.head(12).to_string(index=False))


backtested Subways


backtested Buses


backtested LIRR


backtested Metro-North


backtested Access-A-Ride


backtested Bridges and Tunnels


backtested Staten Island Railway
rows: 490 successful: 490


'transport          model  horizon     origin status            MAE           RMSE   WAPE   sMAPE   MASE\n  Subways          Naive        7 2024-07-21     OK 1,213,245.1429 1,376,985.6077 0.3935 45.8326 5.8946\n  Subways  SeasonalNaive        7 2024-07-21     OK    27,121.5714    37,647.6549 0.0088  0.9726 0.1318\n  Subways MovingAverage7        7 2024-07-21     OK   568,981.8163   665,166.7055 0.1845 19.7181 2.7644\n  Subways          Drift        7 2024-07-21     OK 1,213,663.4348 1,377,563.8717 0.3936 45.8526 5.8966\n  Subways            ETS        7 2024-07-21     OK    44,538.4266    57,086.9663 0.0144  1.3556 0.2164\n  Subways         SARIMA        7 2024-07-21     OK   564,350.4946   592,364.0566 0.1830 19.5207 2.7419\n  Subways        HistGBM        7 2024-07-21     OK    56,065.5763    64,227.4900 0.0182  1.8045 0.2724\n  Subways          Naive        7 2024-07-28     OK 1,179,725.7143 1,337,344.4438 0.3929 45.4905 5.7536\n  Subways  SeasonalNaive        7 2024-07-28     OK   


## 9. Model Evaluation and Comparison

MASE is the primary metric: it scales error by the in-sample weekly seasonal-naive MAE computed only from observations available at each origin. WAPE is reported as a business-volume metric, sMAPE for relative comparison, and RMSE to expose large misses. Models are ranked by mean MASE across backtest origins. The final test period is still untouched at this stage.


In [9]:

aggregated = fu.aggregate_backtest(backtest_df)
selected = fu.select_models(aggregated)
model_comparison = aggregated.merge(
    selected[["transport", "horizon", "selected_model"]], on=["transport", "horizon"], how="left"
)
model_comparison["selected"] = model_comparison["model"] == model_comparison["selected_model"]
model_comparison = model_comparison.sort_values(["transport", "horizon", "MASE"]).reset_index(drop=True)
model_comparison.to_csv(fu.REPORT_DIR / "model_comparison.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 8), sharey=True)
for ax, horizon in zip(axes, fu.HORIZONS):
    sub = model_comparison[model_comparison["horizon"] == horizon]
    pivot = sub.pivot(index="transport", columns="model", values="MASE").reindex(fu.SYSTEM_NAMES)
    pivot.plot(kind="barh", ax=ax, width=0.85)
    ax.axvline(1, color="black", ls="--", lw=1)
    ax.set_title(f"Backtest MASE — {horizon}-day horizon")
    ax.set_xlabel("MASE (lower is better; 1 = seasonal naive)")
    ax.set_ylabel("")
    ax.legend(fontsize=7, ncol=2)
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "06_model_comparison_mase.png", bbox_inches="tight")
plt.close(fig)
display(model_comparison[["transport", "horizon", "model", "MASE", "WAPE", "sMAPE", "MASE_improvement_pct", "selected"]].head(20).to_string(index=False))


'          transport  horizon          model   MASE   WAPE   sMAPE  MASE_improvement_pct  selected\n      Access-A-Ride        7            ETS 0.3557 0.0192  2.0044               17.2426      True\n      Access-A-Ride        7  SeasonalNaive 0.4298 0.0232  2.5143                0.0000     False\n      Access-A-Ride        7        HistGBM 0.8801 0.0477  4.8484             -104.7642     False\n      Access-A-Ride        7         SARIMA 2.4000 0.1299 13.7341             -458.3524     False\n      Access-A-Ride        7 MovingAverage7 2.9971 0.1623 17.0739             -597.2675     False\n      Access-A-Ride        7          Drift 5.4883 0.2972 32.7936           -1,176.8679     False\n      Access-A-Ride        7          Naive 5.4905 0.2974 32.8093           -1,177.3618     False\n      Access-A-Ride       30            ETS 0.7124 0.0388  4.3125                9.2441      True\n      Access-A-Ride       30  SeasonalNaive 0.7850 0.0428  4.7447                0.0000     False\n      Acc


## 10. Residual Diagnostics

Residuals from the selected models' backtests are examined for bias and autocorrelation. Prediction intervals use the empirical 10th and 90th percentiles of those residuals (an 80% additive interval), because several models do not provide reliable native intervals. Coverage is checked once on the held-out test window.


In [10]:

coverage_rows, test_rows = [], []
for _, choice in selected.iterrows():
    name, horizon, model_name = choice["transport"], int(choice["horizon"]), choice["selected_model"]
    s = series[name]
    train = s.iloc[:-horizon]
    actual = s.iloc[-horizon:]
    prediction = fu.safe_forecast(fu.model_registry()[model_name](train, horizon, actual.index))
    metrics = fu.compute_metrics(actual.values, prediction, train.values)
    test_rows.append({"transport": name, "horizon": horizon, "selected_model": model_name,
                      "status": "OK", **metrics})
    lower, centre, upper = fu.residual_intervals(residuals_df, name, model_name, horizon)
    coverage = fu.interval_coverage(actual.values, prediction, lower, upper)
    coverage_rows.append({"transport": name, "horizon": horizon, "model": model_name,
                          "lower_shift": lower, "upper_shift": upper, "coverage_80": coverage})

test_eval = pd.DataFrame(test_rows)
coverage_df = pd.DataFrame(coverage_rows)
test_eval.to_csv(fu.REPORT_DIR / "test_evaluation.csv", index=False)
coverage_df.to_csv(fu.REPORT_DIR / "interval_coverage.csv", index=False)
display(test_eval.to_string(index=False))
display(coverage_df.to_string(index=False))

selected_residuals = residuals_df.merge(
    selected[["transport", "horizon", "selected_model"]],
    left_on=["transport", "horizon", "model"],
    right_on=["transport", "horizon", "selected_model"],
)
fig, axes = plt.subplots(4, 2, figsize=(15, 10), sharex=True)
for ax, name in zip(axes.ravel(), fu.SYSTEM_NAMES):
    values = selected_residuals[(selected_residuals["transport"] == name) & (selected_residuals["horizon"] == 30)]["residual"].dropna()
    ax.hist(values, bins=30, color="#8c564b", alpha=0.8)
    ax.axvline(0, color="black", lw=1)
    ax.set_title(name)
axes.ravel()[-1].axis("off")
fig.suptitle("Selected-model backtest residuals — 30-day horizon")
fig.tight_layout(rect=[0, 0, 1, 0.97])
fig.savefig(fu.VIZ_DIR / "07_residual_diagnostics.png", bbox_inches="tight")
plt.close(fig)


'            transport  horizon selected_model status          MAE         RMSE   WAPE  sMAPE   MASE\n                Buses        7            ETS     OK  61,075.8155  85,247.0306 0.0473 4.7348 0.6555\n              Subways        7  SeasonalNaive     OK 104,370.7143 140,169.0281 0.0281 2.5254 0.5123\nStaten Island Railway        7            ETS     OK     413.3422     479.6249 0.0623 7.5485 0.6986\n        Access-A-Ride        7            ETS     OK     740.7431   1,072.0330 0.0219 2.1633 0.4410\n  Bridges and Tunnels        7        HistGBM     OK  35,912.3385  40,617.8638 0.0376 3.8690 0.8001\n                Buses       30  SeasonalNaive     OK  67,231.0000 105,106.2891 0.0523 5.9018 0.7211\n  Bridges and Tunnels       30        HistGBM     OK  34,173.7335  41,929.7501 0.0360 3.6894 0.7572\n        Access-A-Ride       30            ETS     OK   1,129.2787   1,640.5224 0.0336 3.3461 0.6717\nStaten Island Railway       30  SeasonalNaive     OK     470.5000     693.4340 0.0707 8.83

'            transport  horizon         model   lower_shift  upper_shift  coverage_80\n                Buses        7           ETS  -61,512.6781  30,291.6991       0.5714\n              Subways        7 SeasonalNaive -112,068.6000  75,105.1000       0.4286\nStaten Island Railway        7           ETS     -413.9282     216.2083       0.4286\n        Access-A-Ride        7           ETS     -536.1587   1,286.9611       0.5714\n  Bridges and Tunnels        7       HistGBM  -30,756.3658  32,206.0386       0.5714\n                Buses       30 SeasonalNaive  -90,275.1000  61,122.3000       0.5333\n  Bridges and Tunnels       30       HistGBM  -39,537.8997  46,344.3139       0.7333\n        Access-A-Ride       30           ETS   -1,186.9425   1,285.6720       0.7000\nStaten Island Railway       30 SeasonalNaive     -584.1000     505.1000       0.7000\n              Subways       30           ETS -229,513.6296 130,943.8683       0.5000\n                 LIRR        7           ETS  -10,653


## 11. Final Model Selection

Selection uses backtest MASE only. The held-out test metrics above are reported for transparency, but they did not influence the choice. Where no model beats seasonal naive, the baseline itself is selected and that limitation is stated explicitly.


In [11]:

selection_table = selected.rename(
    columns={"MASE": "validation_MASE", "MASE_improvement_pct": "validation_MASE_improvement_pct"}
)
test_renamed = test_eval.rename(
    columns={"MAE": "test_MAE", "RMSE": "test_RMSE", "WAPE": "test_WAPE",
             "sMAPE": "test_sMAPE", "MASE": "test_MASE"}
)
selection_table = selection_table.merge(
    test_renamed, on=["transport", "horizon", "selected_model"], how="left"
)
selection_table = selection_table.merge(
    coverage_df[["transport", "horizon", "coverage_80"]], on=["transport", "horizon"], how="left"
)
selection_table["beats_seasonal_naive"] = selection_table["validation_MASE_improvement_pct"] > 0
display(selection_table[["transport", "horizon", "selected_model", "validation_MASE",
                         "validation_MASE_improvement_pct", "test_MASE", "test_WAPE",
                         "coverage_80", "beats_seasonal_naive"]].to_string(index=False))

metadata = {
    "generated_from": "WEEK ONE/MTA_Daily_Ridership_Cleaned.csv",
    "history_end": str(series["Subways"].index[-1].date()),
    "test_window_days": fu.TEST_SIZE,
    "horizons": list(fu.HORIZONS),
    "metrics": {"primary": "MASE", "supporting": ["MAE", "RMSE", "WAPE", "sMAPE"]},
    "selected": selection_table[["transport", "horizon", "selected_model", "validation_MASE"]].to_dict("records"),
    "versions": {"python": sys.version.split()[0], "pandas": pd.__version__,
                 "sklearn": sklearn.__version__, "statsmodels": statsmodels.__version__},
}
(fu.MODELS_DIR / "model_metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print("Saved model_metadata.json")


'            transport  horizon selected_model  validation_MASE  validation_MASE_improvement_pct  test_MASE  test_WAPE  coverage_80  beats_seasonal_naive\n                Buses        7            ETS           0.3118                          17.2315     0.6555     0.0473       0.5714                  True\n              Subways        7  SeasonalNaive           0.3386                           0.0000     0.5123     0.0281       0.4286                 False\nStaten Island Railway        7            ETS           0.3468                          19.5704     0.6986     0.0623       0.4286                  True\n        Access-A-Ride        7            ETS           0.3557                          17.2426     0.4410     0.0219       0.5714                  True\n  Bridges and Tunnels        7        HistGBM           0.4749                          21.6639     0.8001     0.0376       0.5714                  True\n                Buses       30  SeasonalNaive           0.6103             

Saved model_metadata.json



## 12. 30-Day Forecasting

Each selected model is refit on the complete history and used to forecast 2024-11-01 through 2024-11-30. Empirical residual intervals are attached. Forecasts are saved per system, and a summary compares the forecast mean with the last 30 observed days.


In [12]:

forecast_parts, summary_rows = [], []
for name, s in series.items():
    choice = selected[(selected["transport"] == name) & (selected["horizon"] == 30)].iloc[0]
    model_name = choice["selected_model"]
    shifts = fu.residual_intervals(residuals_df, name, model_name, 30)
    frame = fu.attach_intervals(fu.final_forecast(s, model_name, 30), shifts)
    frame["selected_model"] = model_name
    frame.to_csv(fu.FORECAST_DIR / f"{fu.slug(name)}_30day_forecast.csv", index=False)
    forecast_parts.append(frame)
    recent_mean = float(s.tail(30).mean())
    forecast_mean = float(frame["point_forecast"].mean())
    summary_rows.append({
        "transport": name,
        "unit": fu.SYSTEM_UNITS[name],
        "selected_model": model_name,
        "forecast_start": frame["date"].min().date(),
        "forecast_end": frame["date"].max().date(),
        "last30_mean": recent_mean,
        "forecast30_mean": forecast_mean,
        "change_pct": 100 * (forecast_mean / recent_mean - 1) if recent_mean > 0 else np.nan,
        "lower80_mean": float(frame["lower_80"].mean()),
        "upper80_mean": float(frame["upper_80"].mean()),
        "validation_MASE": float(choice["MASE"]),
    })

forecast_df = pd.concat(forecast_parts, ignore_index=True)
forecast_summary = pd.DataFrame(summary_rows)
forecast_summary.to_csv(fu.REPORT_DIR / "forecast_summary.csv", index=False)
display(forecast_summary.to_string(index=False))
assert forecast_df["date"].min() > series["Subways"].index[-1]
assert forecast_df["transport"].nunique() == 7
assert len(forecast_df) == 7 * 30
print("PASS: 7 systems x 30 future dates; first forecast is after the last observation.")


'            transport             unit selected_model forecast_start forecast_end    last30_mean  forecast30_mean  change_pct   lower80_mean   upper80_mean  validation_MASE\n              Subways estimated riders            ETS     2024-11-01   2024-11-30 3,641,257.4333   3,643,892.3882      0.0724 3,414,378.7586 3,774,836.2565           0.7532\n                Buses estimated riders  SeasonalNaive     2024-11-01   2024-11-30 1,286,203.9000   1,288,409.9000      0.1715 1,198,134.8000 1,349,532.2000           0.6103\n                 LIRR estimated riders            ETS     2024-11-01   2024-11-30   228,484.2000     223,802.5689     -2.0490   212,186.7525   242,334.2056           1.1725\n          Metro-North estimated riders            ETS     2024-11-01   2024-11-30   204,493.7667     203,787.1629     -0.3455   189,485.7577   216,747.2308           1.0826\n        Access-A-Ride  scheduled trips            ETS     2024-11-01   2024-11-30    33,617.7000      33,691.7298      0.2202    

PASS: 7 systems x 30 future dates; first forecast is after the last observation.



## 13. Forecast Visualization

Historical observations, the train/test boundary, the 30-day forecast and its empirical 80% interval are shown for every system on its own scale. A separate normalized chart compares relative forecast changes across systems without mixing units.


In [13]:

for name, s in series.items():
    frame = forecast_df[forecast_df["transport"] == name]
    fig, ax = plt.subplots(figsize=(12, 4.5))
    ax.plot(s.index, s.values, lw=0.9, color="grey", alpha=0.8, label="Observed")
    ax.axvline(s.index[-fu.TEST_SIZE], color="black", ls="--", lw=1, label="Test boundary")
    ax.plot(frame["date"], frame["point_forecast"], color=fu.w2.SYSTEM_COLORS[name], lw=2, label="Forecast")
    ax.fill_between(frame["date"], frame["lower_80"], frame["upper_80"],
                    color=fu.w2.SYSTEM_COLORS[name], alpha=0.2, label="80% residual interval")
    ax.set_title(f"{name} — 30-day forecast ({fu.SYSTEM_UNITS[name]})")
    ax.set_xlabel("Date")
    ax.set_ylabel(fu.SYSTEM_UNITS[name])
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    fig.tight_layout()
    fig.savefig(fu.VIZ_DIR / f"08_forecast_{fu.slug(name)}.png", bbox_inches="tight")
    plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
values = forecast_summary.set_index("transport")["change_pct"].reindex(fu.SYSTEM_NAMES)
order = values.sort_values()
ax.barh(order.index, order.values, color="#4c78a8")
ax.axvline(0, color="black", lw=0.8)
ax.set_title("Forecast mean versus last 30 observed days (unit-free change)")
ax.set_xlabel("Change (%)")
ax.set_ylabel("")
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "09_forecast_comparison_normalized.png", bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
indexed = forecast_summary.set_index("transport")
ax.bar(indexed.index, indexed["validation_MASE"], color="#f58518")
ax.axhline(1, color="black", ls="--", lw=1, label="Seasonal naive = 1")
ax.set_title("Validation MASE of selected models by system")
ax.set_xlabel("System")
ax.set_ylabel("MASE")
ax.tick_params(axis="x", rotation=30)
ax.legend()
fig.tight_layout()
fig.savefig(fu.VIZ_DIR / "10_selected_model_mase.png", bbox_inches="tight")
plt.close(fig)



## 14. Conclusions and Recommendations

Forecast reliability differs materially by system. Systems with strong weekly structure and modest volatility should be trusted more than intermittent or structurally unstable series. Selection is based on backtest MASE, test metrics are reported separately, and prediction intervals are empirical rather than guaranteed ranges. Any forecast used operationally should be refreshed when new observations arrive and monitored against realised MASE.


In [14]:

print("Selected 30-day models:")
for _, row in forecast_summary.iterrows():
    print(f"  {row['transport']}: {row['selected_model']} | validation MASE {row['validation_MASE']:.3f} | "
          f"forecast mean {row['forecast30_mean']:,.0f} {row['unit']} ({row['change_pct']:+.1f}% vs last 30 days)")
print("\nArtifacts:")
print(f"  {len(list(fu.VIZ_DIR.glob('*.png')))} visualizations")
print(f"  {len(list(fu.FORECAST_DIR.glob('*.csv')))} forecast CSVs")
print(f"  {len(list(fu.REPORT_DIR.glob('*.csv')))} report CSVs")


Selected 30-day models:
  Subways: ETS | validation MASE 0.753 | forecast mean 3,643,892 estimated riders (+0.1% vs last 30 days)
  Buses: SeasonalNaive | validation MASE 0.610 | forecast mean 1,288,410 estimated riders (+0.2% vs last 30 days)
  LIRR: ETS | validation MASE 1.172 | forecast mean 223,803 estimated riders (-2.0% vs last 30 days)
  Metro-North: ETS | validation MASE 1.083 | forecast mean 203,787 estimated riders (-0.3% vs last 30 days)
  Access-A-Ride: ETS | validation MASE 0.712 | forecast mean 33,692 scheduled trips (+0.2% vs last 30 days)
  Bridges and Tunnels: HistGBM | validation MASE 0.667 | forecast mean 922,527 vehicles (-2.8% vs last 30 days)
  Staten Island Railway: SeasonalNaive | validation MASE 0.730 | forecast mean 6,562 estimated riders (-1.4% vs last 30 days)

Artifacts:
  17 visualizations
  7 forecast CSVs
  5 report CSVs
